# ⏳ Notebook 2: Time-Bounded Leases

A **lease** is a lock with an expiration time. The holder must *renew* the lease before it expires; otherwise it's automatically released and someone else can grab it. Used by Chubby, etcd, ZooKeeper, Kubernetes leader election.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/lease
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟨 BETTER: a leased lock

In [ ]:
import time
from dataclasses import dataclass
from typing import Optional

@dataclass
class Lease:
    holder: Optional[str] = None
    expires_at: float = 0.0

    def _expired(self):
        return time.monotonic() >= self.expires_at

    def acquire(self, who: str, ttl: float) -> bool:
        if self.holder is None or self._expired():
            self.holder = who
            self.expires_at = time.monotonic() + ttl
            return True
        return False

    def renew(self, who: str, ttl: float) -> bool:
        if self.holder == who and not self._expired():
            self.expires_at = time.monotonic() + ttl
            return True
        return False

    def release(self, who: str):
        if self.holder == who:
            self.holder = None
            self.expires_at = 0.0


## ⚡ Scenario: holder crashes, lease expires, someone else takes over

In [ ]:
lease = Lease()
assert lease.acquire('A', ttl=1.0) is True
assert lease.acquire('B', ttl=1.0) is False, 'B must not steal a live lease'
print('A acquires (ttl=1s): True')
print('B tries immediately: False')

time.sleep(1.1)  # A 'crashed' and never renewed

assert lease.acquire('B', ttl=1.0) is True, 'lease should have expired on its own'
assert lease.holder == 'B'
print('B tries after expiry: True')
print('holder now:', lease.holder)
print('\n✔ the lock released itself without anyone having to declare A dead')


## 🔄 Healthy holder renews before expiry

In [ ]:
lease = Lease()
lease.acquire('A', ttl=0.5)
for i in range(3):
    time.sleep(0.3)
    ok = lease.renew('A', ttl=0.5)
    assert ok, 'a healthy holder renewing every 0.3s of a 0.5s TTL must never lose it'
    assert lease.acquire('B', ttl=0.5) is False, 'B grabbed a lease that was being renewed'
    print(f'  renew #{i}: {ok}, holder={lease.holder}')
print('✔ renewing at TTL/2 kept the lease continuously held')


## 🕳️ Demonstrating the failure: two holders at once

We give the grantor and the holder separate clocks and let the holder's run **slow** — a
laptop whose clock drifts, a VM that got descheduled, a process stuck in a stop-the-world GC
pause. No `sleep`, no flakiness: both clocks are explicit numbers we advance by hand.

The question we ask at every instant is the one that decides whether the lease is safe:
*is there a moment when two different processes both believe they hold it?*

In [ ]:
class LeaseServer:
    """The grantor. Its clock is the only one that decides who owns the lease."""
    def __init__(self):
        self.now = 0.0
        self.holder = None
        self.expires_at = 0.0
        self.epoch = 0            # fencing token: bumped on every NEW grant

    def acquire(self, who, ttl):
        if self.holder is None or self.now >= self.expires_at:
            self.holder, self.expires_at = who, self.now + ttl
            self.epoch += 1
            return self.epoch
        return None


class Holder:
    """A process that thinks it holds the lease until its OWN clock says otherwise."""
    def __init__(self, name, drift=1.0, safety_margin=0.0):
        self.name = name
        self.now = 0.0
        self.drift = drift              # 1.0 = perfect; 0.8 = runs 20% slow
        self.safety_margin = safety_margin
        self.deadline = None
        self.epoch = None

    def granted(self, epoch, ttl):
        # The holder can only start its own timer when it hears back, and it must
        # trust its own clock from then on.
        self.deadline = self.now + ttl - self.safety_margin
        self.epoch = epoch

    def believes_it_holds(self):
        return self.deadline is not None and self.now < self.deadline

    def tick(self, real_dt):
        self.now += real_dt * self.drift


TTL, STEP, HORIZON = 10.0, 0.1, 20.0

def run(drift, safety_margin=0.0):
    """Return (overlap_seconds, first_overlap_time) for a holder whose clock drifts."""
    server = LeaseServer()
    a = Holder('A', drift=drift, safety_margin=safety_margin)
    b = Holder('B', drift=1.0, safety_margin=safety_margin)
    a.granted(server.acquire('A', TTL), TTL)

    overlap, first = 0.0, None
    t = 0.0
    while t < HORIZON:
        t += STEP
        server.now += STEP
        a.tick(STEP); b.tick(STEP)
        # B is a standby: it keeps trying, and succeeds the moment the server expires A.
        if not b.believes_it_holds():
            tok = server.acquire('B', TTL)
            if tok is not None:
                b.granted(tok, TTL)
        if a.believes_it_holds() and b.believes_it_holds():
            overlap += STEP
            if first is None:
                first = round(t, 2)
    return round(overlap, 2), first


perfect = run(drift=1.0)
skewed  = run(drift=0.8)     # A's clock runs 20% slow
print(f"A's clock perfect  -> overlap = {perfect[0]:.1f}s")
print(f"A's clock 20% slow -> overlap = {skewed[0]:.1f}s, starting at t={skewed[1]}s")

# With matched clocks there is no overlap at all. Skew alone — no crash, no partition,
# no bug — is enough to produce two simultaneous holders.
assert skewed[0] > 1.0, skewed
print(f'\n💥 for {skewed[0]:.1f} seconds, A and B BOTH believe they hold the lease.')
print('   A is not buggy and it is not lying. Its clock is just slow.')

### The two ways out

There are exactly two, and real systems use one or both:

**1. A bounded clock assumption.** Decide a maximum skew + round-trip you will tolerate, and
have the holder give the lease up that much *early*. Then the holder always stops before the
grantor reassigns. This is what Chubby and etcd sessions do — and it is an *assumption*: if
reality exceeds the budget, safety is gone with no warning.

**2. A fencing token.** Give up on clocks entirely for safety and make the **resource** reject
writes from an old epoch. Then two holders overlapping is harmless: only one of them can
actually do anything. This is the only option that survives an unbounded pause.

In [ ]:
# --- Option 1: a safety margin big enough to cover the drift -----------------------
# A's clock loses 20% of 10s = 2s over a TTL, so it must self-expire >2s early.
for margin in (0.0, 1.0, 2.0, 3.0):
    ov, _ = run(drift=0.8, safety_margin=margin)
    print(f'safety_margin={margin:>4.1f}s -> overlap {ov:>4.1f}s')

assert run(drift=0.8, safety_margin=3.0)[0] == 0.0, 'a margin above the real skew must eliminate overlap'
assert run(drift=0.8, safety_margin=1.0)[0] > 0.0, 'a margin below the real skew must NOT be enough'

# ...but the assumption is load-bearing. Make the drift worse than budgeted and it breaks.
assert run(drift=0.5, safety_margin=3.0)[0] > 0.0, 'expected the budget to be blown'
print('\n⚠️ a 3s margin survives 20% drift and fails at 50% drift — the safety is only')
print('   as good as the bound you assumed, and nothing tells you when it is exceeded.')

In [ ]:
# --- Option 2: fencing. Overlap still happens; it just stops mattering. -------------
class FencedResource:
    def __init__(self):
        self.highest_epoch = 0
        self.log = []

    def write(self, who, value, epoch):
        if epoch < self.highest_epoch:          # strictly older leader -> refuse
            return False
        self.highest_epoch = epoch
        self.log.append((who, epoch, value))
        return True


server = LeaseServer()
res = FencedResource()
a = Holder('A', drift=0.8)
b = Holder('B', drift=1.0)
a.granted(server.acquire('A', TTL), TTL)

both_believed = False
t = 0.0
while t < HORIZON:
    t += STEP
    server.now += STEP
    a.tick(STEP); b.tick(STEP)
    if not b.believes_it_holds():
        tok = server.acquire('B', TTL)
        if tok is not None:
            b.granted(tok, TTL)
    # Both write whenever they think they are in charge.
    for h in (a, b):
        if h.believes_it_holds():
            res.write(h.name, f'{h.name}-work@{t:.1f}', h.epoch)
    if a.believes_it_holds() and b.believes_it_holds():
        both_believed = True

assert both_believed, 'the overlap should still be there — fencing does not prevent it'
epochs_that_wrote = sorted({e for _, e, _ in res.log})
writers_after_b   = {w for w, e, _ in res.log if e == max(epochs_that_wrote)}

# The invariant fencing actually gives you: once epoch 2 has written, epoch 1 can never
# write again. Two believers, one writer.
first_epoch2 = next(i for i, (_, e, _) in enumerate(res.log) if e == 2)
assert all(e == 2 for _, e, _ in res.log[first_epoch2:]), 'a fenced leader wrote after the handover'
assert writers_after_b == {'B'}
print(f'overlap still occurred: {both_believed}')
print(f'epochs that reached the resource: {epochs_that_wrote}')
print(f'\n✔ A kept believing and kept trying, but every write it made after B took over')
print(f'  was refused. Correctness no longer depends on whose clock is right.')

## ⚠️ The unsafe part: whose clock?

Everything above used **one** clock, because the `Lease` object is the grantor *and* the
thing being asked "am I still the holder?". Real deployments have two clocks: the lease
service's, and the holder's. They do not agree.

A lease is only safe if the holder stops acting **before** the grantor hands the lease to
someone else. Nothing in the code above enforces that. Let's break it.

## 🧠 Comparison

| | Plain lock | Lease |
|---|---|---|
| Holder crashes | locked forever | auto-released after TTL |
| Cost while healthy | none | periodic renew RPC |
| Tunable safety | n/a | shorter TTL = faster recovery, more renew traffic |

## ⚠️ Gotchas

- **Clock skew** between nodes can let a 'live' holder think it still has the lease while a 'fresh' holder also thinks it does. → see the *split-brain-and-fencing* lab for **fencing tokens** that fix this.
- TTL should be **>> max network round-trip + GC pause** to avoid spurious expirations.
